In [1]:
# calculate mutants of a79b73/a79b71
# test ensembly of original model

In [2]:
import numpy as np
import pandas as pd
import csv
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
from sklearn.metrics import roc_auc_score
import json
import subprocess
import math
from prefetch_generator import BackgroundGenerator, background
from pandarallel import pandarallel

In [3]:
print(torch.__version__)

2.3.0


In [4]:
print(torch.version.cuda)

12.1


In [5]:
CUDA = 0
SEED = 50

ESM_FILE = "esm2_650m_out"
ESM_DIM = 1280
ESM_LAYER = 33
#VAL_BATCH_SIZE = 50000


MODEL_LOADED=["/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy3_Epoch20.pth",
              "/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy8_Epoch20.pth",
              "/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy9_Epoch30.pth",
              "/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy13_Epoch19.pth",
              "/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy14_Epoch47.pth",
              "/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/tmp_model3/pmhctcr_train_tcr_pmhc_external_Copy15_Epoch20.pth"]

ORIGINAL_MODEL=["/project/DPDS/Wang_lab/shared/pMTnet_v2/code/saved_model_tmp/pmhctcrCopy240_stage13_Epoch5.pth",
               "/project/DPDS/Wang_lab/shared/pMTnet_v2/code/saved_model_tmp/pmhctcrCopy241_stage13_Epoch5.pth",
               "/project/DPDS/Wang_lab/shared/pMTnet_v2/code/saved_model_tmp/pmhctcrCopy242_stage13_Epoch5.pth"]

VAL_BATCH_SIZE_bgTCR = 20000
VAL_BATCH_SIZE_bgPMHC = 20000

#N_BACKGROUND_TCR_ONLY = 5000
#N_BACKGROUND_PMHC_ONLY = 5

N_BACKGROUND_TCR_ONLY = 2000
N_BACKGROUND_PMHC_ONLY = 2000

N_BACKGROUND_TCR_ONLY4 = 100000
N_BACKGROUND_PMHC_ONLY4 = 10000

N_BACKGROUND_TCR_ONLY5 = 100000
N_BACKGROUND_PMHC_ONLY5 = 10000

READ_MHC = "reduce"


VAL_PATH1 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_tcr_val5neg_train50neg_maxEpoch70/validation.txt"
VAL_PATH2_1 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_pmhc_val5neg_train50neg_maxEpoch70/validation.txt"
VAL_PATH2_2 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_p_val5neg_train50neg_maxEpoch70/validation.txt"
VAL_PATH3 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_pmhc_tcr_val5neg_train50neg_maxEpoch70/validation.txt"
VAL_PATH4 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/feb23_1tcr_more_pmhc_pseudoVgene_upHc2Mouse_val5neg_train50neg_maxEpoch70/validation_hormad1.txt"
VAL_PATH5 = "/project/DPDS/Wang_lab/shared/pMTnet_v2/data/koelle/Koelle_input.csv"

In [6]:
pandarallel.initialize(progress_bar=False)

INFO: Pandarallel will run on 36 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [7]:
def setup_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

In [8]:
setup_seed(SEED)

In [9]:
if READ_MHC == "reduce":
    def load_dic(filename):
        with open(filename) as f:
            dic = json.loads(f.read())
        dic_tensor = dict((k, torch.from_numpy(np.array(dic[k]))) for k in dic.keys())
        return dic_tensor
    mhc_dic = load_dic("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/ipd_imgt/" + ESM_FILE + "_reduced_numpy_include_iag7.txt")

elif READ_MHC == "full":
    directory = os.fsencode("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/ipd_imgt/" + ESM_FILE)
    mhc_dic = {}
    for file in os.listdir(directory):
        filename = os.fsdecode(file)
        esm_file = torch.load(os.path.join("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/ipd_imgt/" + ESM_FILE, filename))
        mhc_dic[filename.split(".")[0]] = esm_file["representations"][ESM_LAYER]

In [10]:
aa_dict_atchley=dict()
aa_dict_dir='/project/DPDS/Wang_lab/shared/pMTnet_v2/data/pmtnetv1/Atchley_factors.csv'
with open(aa_dict_dir,'r') as aa:
    aa_reader=csv.reader(aa)      
    next(aa_reader, None)
    for rows in aa_reader:
        aa_name=rows[0]
        aa_factor=rows[1:len(rows)]
        aa_dict_atchley[aa_name]=np.asarray(aa_factor,dtype='float')

In [11]:
def peptideMap(dataset, aa_dict, column, padding):
    peptideArray = np.zeros((len(dataset), 1, padding, 5), dtype=np.float32)
    for pos, seq in enumerate(dataset[column]):
        peptideArray[pos, 0] = aamapping(seq, aa_dict, padding)
    return peptideArray

In [12]:
def aamapping(peptideSeq, aa_dict, padding):
    peptideArray = []
    if len(peptideSeq)>padding:
        #print('Length: '+str(len(peptideSeq))+'is over bound'+ ' (' +str(padding)+ ')' +'!')
        peptideSeq=peptideSeq[0:padding]
    for aa_single in peptideSeq:
        try:
            peptideArray.append(aa_dict[aa_single])
        except:
#            print('Inproper aa: ' + aa_single + ', in seq: ' + peptideSeq + '. 0 was applied for encoding.')
            peptideArray.append(np.zeros(5, dtype='float32'))
    return np.concatenate((np.asarray(peptideArray), np.zeros((padding - len(peptideSeq), 5), dtype='float32')), axis=0)

In [13]:
def mhcMap(dataset, allele, mhc_dic):
    mhc_array = np.zeros((len(dataset), 1, 380, ESM_DIM), dtype=np.float32)
    mhc_seen = dict()
    for pos, mhc in enumerate(dataset[allele]):
        try:
            mhc_array[pos, 0] = mhc_seen[mhc]
        except:
            mhc_array[pos, 0] = esmmapping(mhc,mhc_dic)
            mhc_seen[mhc] = mhc_array[pos, 0]
    return mhc_array

In [14]:
def esmmapping(mhc,mhc_dic):
    mhc_encoding = mhc_dic[mhc].numpy()
    num_padding = 380-mhc_encoding.shape[0]
    return np.concatenate((mhc_encoding, np.zeros((num_padding,ESM_DIM),dtype='float32')), axis=0)

In [15]:
def preprocess(filedir, mhc_dic, a_allele, b_allele, sepd):
    #1. input file path is valid or not
    print('Processing: '+filedir)
    if not os.path.exists(filedir):
        print('Invalid file path: ' + filedir)
        return 0
    dataset = pd.read_csv(filedir, header=0, sep=sepd)
    print("Number of rows in raw dataset: " + str(dataset.shape[0]))
    
    #mhc_dic_keys = set(mhc_dic.keys())
    #dataset = dataset[dataset[a_allele].isin(mhc_dic_keys)]
    #dataset = dataset[dataset[b_allele].isin(mhc_dic_keys)]
    
    dataset[dataset['mhca'].isin(SAVED_PMHC['mhca'])]
    dataset[dataset['mhcb'].isin(SAVED_PMHC['mhcb'])]

    dataset = dataset.reset_index(drop=True)
    print("Number of rows in processed dataset: " + str(dataset.shape[0]))
    return dataset

In [16]:
def preprocess_validation(filedir,sep_d):
    #1. input file path is valid or not
    print('Processing: '+filedir)
    if not os.path.exists(filedir):
        print('Invalid file path: ' + filedir)
        return 0
    dataset = pd.read_csv(filedir, header=0, sep=sep_d)
    print("Number of rows in raw dataset: " + str(dataset.shape[0]))
    
    #mhc_dic_keys = set(mhc_dic.keys())
    
    dataset[dataset['mhca'].isin(SAVED_PMHC['mhca'])]
    dataset[dataset['mhcb'].isin(SAVED_PMHC['mhcb'])]

    dataset = dataset.reset_index(drop=True)
    print("Number of rows in processed dataset: " + str(dataset.shape[0]))
    return dataset

In [17]:
def preprocessTCR(filedir):
    #1. input file path is valid or not
    #print('Processing: '+filedir)
    if not os.path.exists(filedir):
        print('Invalid file path: ' + filedir)
        return 0
    dataset = pd.read_csv(filedir, header=0, sep="\t")
    print("Number of rows in raw dataset: " + str(dataset.shape[0]))
    dataset=dataset.dropna()
    print("Number of rows in this dataset after dropping NA: " + str(dataset.shape[0]))
    dataset = dataset.reset_index(drop=True)
    print("Number of rows in processed dataset: " + str(dataset.shape[0]))
    return dataset

In [18]:
def preprocesspMHC2(filedir):
    #1. input file path is valid or not
    #print('Processing: '+filedir)
    if not os.path.exists(filedir):
        print('Invalid file path: ' + filedir)
        return 0
    dataset = pd.read_csv(filedir, header=0, sep=",")
    print("Number of rows in raw dataset: " + str(dataset.shape[0]))
    dataset=dataset.dropna()
    print("Number of rows in this dataset after dropping NA: " + str(dataset.shape[0]))
    
    dataset[dataset['alpha_allele'].isin(SAVED_PMHC['mhca'])]
    dataset[dataset['beta_allele'].isin(SAVED_PMHC['mhcb'])]

    
    dataset = dataset.reset_index(drop=True)
    print("Number of rows in processed dataset: " + str(dataset.shape[0]))
    return dataset

In [19]:
def add_background_tcr(val_data, human_alpha_tcr, human_beta_tcr, mouse_alpha_tcr, mouse_beta_tcr, fold):
    
    type_col = pd.Series(['validation'], name="_data_type") # validation | background
    
    val_data_com = pd.concat([val_data.reset_index(drop=True),type_col.repeat(len(val_data)).reset_index(drop=True)],axis=1)
    
    val_data_com_htcr = val_data_com.loc[val_data_com['TCR_SPECIES']=="human"].reset_index(drop=True)
    val_data_com_mtcr = val_data_com.loc[val_data_com['TCR_SPECIES']=="mouse"].reset_index(drop=True)
    
    label_col = pd.Series([-1], name="label")
    type_col2 = pd.Series(['background'], name="_data_type")
    
    
    #human TCR
    val_mhc_htcr_tmp = val_data_com_htcr.drop(columns=['cdr3a','cdr3b','vaseq','vbseq','_data_type','va','vb'], errors='ignore').drop_duplicates(subset="_pmhc").reset_index(drop=True)
    val_mhc_htcr = val_mhc_htcr_tmp.loc[np.repeat(val_mhc_htcr_tmp.index, fold)].reset_index(drop=True)
        
    if 'label' in val_mhc_htcr.columns:
        val_mhc_htcr = val_mhc_htcr.drop(columns=['label'])
        
        val_background_htcr = pd.concat([val_mhc_htcr, human_alpha_tcr.sample(n=len(val_mhc_htcr),replace=True).reset_index(drop=True), human_beta_tcr.sample(n=len(val_mhc_htcr),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_htcr)).reset_index(drop=True),label_col.repeat(len(val_mhc_htcr)).reset_index(drop=True)],axis=1)
    else:
        
        val_background_htcr = pd.concat([val_mhc_htcr, human_alpha_tcr.sample(n=len(val_mhc_htcr),replace=True).reset_index(drop=True), human_beta_tcr.sample(n=len(val_mhc_htcr),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_htcr)).reset_index(drop=True)],axis=1)


    
    #mouse TCR
    val_mhc_mtcr_tmp = val_data_com_mtcr.drop(columns=['cdr3a','cdr3b','vaseq','vbseq','_data_type','va','vb'], errors='ignore').drop_duplicates(subset="_pmhc").reset_index(drop=True)
    val_mhc_mtcr = val_mhc_mtcr_tmp.loc[np.repeat(val_mhc_mtcr_tmp.index, fold)].reset_index(drop=True)
    
    if 'label' in val_mhc_mtcr.columns:
        val_mhc_mtcr = val_mhc_mtcr.drop(columns=['label'])
        
        val_background_mtcr = pd.concat([val_mhc_mtcr, mouse_alpha_tcr.sample(n=len(val_mhc_mtcr),replace=True).reset_index(drop=True), mouse_beta_tcr.sample(n=len(val_mhc_mtcr),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_mtcr)).reset_index(drop=True),label_col.repeat(len(val_mhc_mtcr)).reset_index(drop=True)],axis=1)
    else:
        val_background_mtcr = pd.concat([val_mhc_mtcr, mouse_alpha_tcr.sample(n=len(val_mhc_mtcr),replace=True).reset_index(drop=True), mouse_beta_tcr.sample(n=len(val_mhc_mtcr),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_mtcr)).reset_index(drop=True)],axis=1)
    
    
    val_background = pd.concat([val_data_com_htcr, val_data_com_mtcr, val_background_htcr, val_background_mtcr])
    
    #print("val_background:")
    #print(val_background.shape[0])
    
    return(val_background.reset_index(drop=True))

In [20]:
def add_background_pmhc(val_data, human_pmhc, mouse_pmhc, fold):
    
    type_col = pd.Series(['validation'], name="_data_type") # validation | background
    
    val_data_com = pd.concat([val_data.reset_index(drop=True),type_col.repeat(len(val_data)).reset_index(drop=True)],axis=1)
    
    val_data_com_hpmhc = val_data_com.loc[val_data_com['pMHC_SPECIES']=="human"].reset_index(drop=True)
    val_data_com_mpmhc = val_data_com.loc[val_data_com['pMHC_SPECIES']=="mouse"].reset_index(drop=True)
    
    label_col = pd.Series([-1], name="label")
    type_col2 = pd.Series(['background'], name="_data_type")
    
    
    #human pmhc
    val_mhc_hpmhc_tmp = val_data_com_hpmhc.drop(columns=['peptide','mhca','mhcb','_data_type'], errors='ignore').drop_duplicates(subset="_tcr").reset_index(drop=True)
    val_mhc_hpmhc = val_mhc_hpmhc_tmp.loc[np.repeat(val_mhc_hpmhc_tmp.index, fold)].reset_index(drop=True)
        
    if 'label' in val_mhc_hpmhc.columns:
        val_mhc_hpmhc = val_mhc_hpmhc.drop(columns=['label'])
        
        val_background_hpmhc = pd.concat([val_mhc_hpmhc, human_pmhc.sample(n=len(val_mhc_hpmhc),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_hpmhc)).reset_index(drop=True),label_col.repeat(len(val_mhc_hpmhc)).reset_index(drop=True)],axis=1)
    else:
        
        val_background_hpmhc = pd.concat([val_mhc_hpmhc, human_pmhc.sample(n=len(val_mhc_hpmhc),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_hpmhc)).reset_index(drop=True)],axis=1)


    
    #mouse pmhc
    val_mhc_mpmhc_tmp = val_data_com_mpmhc.drop(columns=['peptide','mhca','mhcb','_data_type'], errors='ignore').drop_duplicates(subset="_tcr").reset_index(drop=True)
    val_mhc_mpmhc = val_mhc_mpmhc_tmp.loc[np.repeat(val_mhc_mpmhc_tmp.index, fold)].reset_index(drop=True)
    
    if 'label' in val_mhc_mpmhc.columns:
        val_mhc_mpmhc = val_mhc_mpmhc.drop(columns=['label'])
        
        val_background_mpmhc = pd.concat([val_mhc_mpmhc, mouse_pmhc.sample(n=len(val_mhc_mpmhc),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_mpmhc)).reset_index(drop=True),label_col.repeat(len(val_mhc_mpmhc)).reset_index(drop=True)],axis=1)
    else:
        val_background_mpmhc = pd.concat([val_mhc_mpmhc, mouse_pmhc.sample(n=len(val_mhc_mpmhc),replace=True).reset_index(drop=True),
                                     type_col2.repeat(len(val_mhc_mpmhc)).reset_index(drop=True)],axis=1)
    
    val_background = pd.concat([val_data_com_hpmhc, val_data_com_mpmhc, val_background_hpmhc, val_background_mpmhc])

    return(val_background.reset_index(drop=True))

In [21]:
# model 1
class pMHC(nn.Module):
    def __init__(self):
        super(pMHC, self).__init__()
        self.layerP1 = nn.Sequential(
            nn.Conv2d(1, 200,(2,5)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(2,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(2,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((30-3*2+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerP2 = nn.Sequential(
            nn.Conv2d(1, 200,(4,5)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(4,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(4,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((30-3*4+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerP3 = nn.Sequential(
            nn.Conv2d(1, 200,(6,5)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(6,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(6,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((30-3*6+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerA1 = nn.Sequential(
            nn.Conv2d(1, 200,(10,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(10,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(10,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*10+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerA2 = nn.Sequential(
            nn.Conv2d(1, 200,(20,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(20,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(20,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*20+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerA3 = nn.Sequential(
            nn.Conv2d(1, 200,(30,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(30,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(30,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*30+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerB1 = nn.Sequential(
            nn.Conv2d(1, 200,(10,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(10,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(10,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*10+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerB2 = nn.Sequential(
            nn.Conv2d(1, 200,(20,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(20,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(20,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*20+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.layerB3 = nn.Sequential(
            nn.Conv2d(1, 200,(30,ESM_DIM)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(30,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200,(30,1)),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d((380-3*30+3*1,1)),
            nn.Flatten(),
            nn.Linear(200, int(200/2)),
            nn.ReLU(),
            nn.Linear(int(200/2), 3)
        )
        self.fc1 = nn.Linear(3 * 9, 30)
        self.fc2 = nn.Linear(30, 3)
    def forward(self, x_p, x_a, x_b):
        f1_p = self.layerP1(x_p)
        f2_p = self.layerP2(x_p)
        f3_p = self.layerP3(x_p)
        
        f1_a = self.layerA1(x_a)
        f2_a = self.layerA2(x_a)
        f3_a = self.layerA3(x_a)
        
        f1_b = self.layerB1(x_b)
        f2_b = self.layerB2(x_b)
        f3_b = self.layerB3(x_b)
        encoded = self.fc1(torch.cat((f1_p,f2_p,f3_p, f1_a,f2_a,f3_a, f1_b,f2_b,f3_b),dim=1))
        encoded_act = F.relu(encoded)
        return encoded_act, self.fc2(encoded_act)

In [22]:
# 2. V gene alpha model
class vGdVAEa(nn.Module):
    def __init__(self):
        super(vGdVAEa, self).__init__()
        self.layer1 = nn.Sequential(  
            nn.Conv2d(1, 180,(10,5)),
            nn.ReLU(),   
            nn.MaxPool2d((100-10+1,1)),
            nn.Flatten(), 
            nn.Linear(180, int(180/2)),    
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(1, 180,(20,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-20+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(1, 180,(30,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-30+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(1, 180,(40,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-40+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(1, 180,(50,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-50+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer6 = nn.Sequential(
            nn.Conv2d(1, 180,(60,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-60+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer7 = nn.Sequential(
            nn.Conv2d(1, 180,(70,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-70+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer8 = nn.Sequential(
            nn.Conv2d(1, 180,(80,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-80+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer9 = nn.Sequential(
            nn.Conv2d(1, 180,(90,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-90+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer10 = nn.Sequential(
            nn.Conv2d(1, 180,(100,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-100+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=5, nhead=5), num_layers=6)
        self.linear1 = nn.Linear(in_features=3*10, out_features=int(3*10/2))
        self.linear2 = nn.Linear(in_features=int(3*10/2), out_features=5)
        self.linear3 = nn.Linear(in_features=int(3*10/2), out_features=5)
        self.decoder = nn.Sequential(
            nn.Linear(in_features=5, out_features=int(180*10/2)),
            nn.ReLU(),
            nn.Linear(in_features=int(180*10/2), out_features=180*10),
            nn.Unflatten(1,(10,180,1)),
            nn.Conv2d(in_channels=10, out_channels=1, kernel_size=(180-100+1,5), padding=(0,4))
        )
    def forward(self, x):
        padding_mask = torch.sum(x.squeeze(dim=1),dim=2) == 0
        x = x.permute(2,1,0,3)
        x = torch.squeeze(x,dim=1)
        x = self.transformer_encoder(src = x,src_key_padding_mask = padding_mask)
        x = torch.unsqueeze(x,dim=1)
        x = x.permute(2,1,0,3)
        f1 = self.layer1(x)
        f2 = self.layer2(x)
        f3 = self.layer3(x)
        f4 = self.layer4(x)
        f5 = self.layer5(x)
        f6 = self.layer6(x)
        f7 = self.layer7(x)
        f8 = self.layer8(x)
        f9 = self.layer9(x)
        f10 = self.layer10(x)
        h1 = F.relu(self.linear1(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10),1)))
        mu = self.linear2(h1)
        #logvar = self.linear3(h1)
        #std = torch.exp(0.5*logvar)
        #eps = torch.randn_like(std)
        #encoded = mu + eps*std
        #encoded = mu
        #decoded = self.decoder(encoded)
        #return encoded, decoded, mu, logvar
        return mu

In [23]:
# 3 v gene beta model
class vGdVAEb(nn.Module):
    def __init__(self):
        super(vGdVAEb, self).__init__()   
        self.layer1 = nn.Sequential(  
            nn.Conv2d(1, 180,(10,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-10+1,1)),
            nn.Flatten(), 
            nn.Linear(180, int(180/2)),    
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(1, 180,(20,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-20+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(1, 180,(30,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-30+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(1, 180,(40,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-40+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(1, 180,(50,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-50+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer6 = nn.Sequential(
            nn.Conv2d(1, 180,(60,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-60+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer7 = nn.Sequential(
            nn.Conv2d(1, 180,(70,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-70+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer8 = nn.Sequential(
            nn.Conv2d(1, 180,(80,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-80+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer9 = nn.Sequential(
            nn.Conv2d(1, 180,(90,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-90+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.layer10 = nn.Sequential(
            nn.Conv2d(1, 180,(100,5)),
            nn.ReLU(),
            nn.MaxPool2d((100-100+1,1)),
            nn.Flatten(),
            nn.Linear(180, int(180/2)),
            nn.ReLU(),
            nn.Linear(int(180/2), 3)
        )
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=5, nhead=5), num_layers=6)
        self.linear1 = nn.Linear(in_features=3*10, out_features=int(3*10/2))
        self.linear2 = nn.Linear(in_features=int(3*10/2), out_features=5)
        self.linear3 = nn.Linear(in_features=int(3*10/2), out_features=5)
        self.decoder = nn.Sequential(
            nn.Linear(in_features=5, out_features=int(180*10/2)),
            nn.ReLU(),
            nn.Linear(in_features=int(180*10/2), out_features=180*10),
            nn.Unflatten(1,(10,180,1)),
            nn.Conv2d(in_channels=10, out_channels=1, kernel_size=(180-100+1,5), padding=(0,4))
        )
    def forward(self, x):
        padding_mask = torch.sum(x.squeeze(dim=1),dim=2) == 0
        x = x.permute(2,1,0,3)
        x = torch.squeeze(x,dim=1)
        x = self.transformer_encoder(src = x,src_key_padding_mask = padding_mask)
        x = torch.unsqueeze(x,dim=1)
        x = x.permute(2,1,0,3)
        f1 = self.layer1(x)
        f2 = self.layer2(x)
        f3 = self.layer3(x)
        f4 = self.layer4(x)
        f5 = self.layer5(x)
        f6 = self.layer6(x)
        f7 = self.layer7(x)
        f8 = self.layer8(x)
        f9 = self.layer9(x)
        f10 = self.layer10(x)
        h1 = F.relu(self.linear1(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10),1)))
        mu = self.linear2(h1)
        #logvar = self.linear3(h1)
        #std = torch.exp(0.5*logvar)
        #eps = torch.randn_like(std)
        #encoded = mu + eps*std
        #encoded = mu
        #decoded = self.decoder(encoded)
        #return encoded, decoded, mu, logvar
        return mu

In [24]:
# 4 CDR3 alpha model
class cdr3VAEa(nn.Module):
    def __init__(self):
        super(cdr3VAEa, self).__init__()
        self.layer1 = nn.Sequential(  
            nn.Conv2d(1, 150,(1,5)),
            nn.ReLU(),   
            nn.MaxPool2d((25,1)),
            nn.Flatten(), 
            nn.Linear(150, int(150/2)),    
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(1, 150,(2,5)),
            nn.ReLU(),
            #nn.BatchNorm2d(30),
            nn.MaxPool2d((25-2+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(1, 150,(3,5)),
            nn.ReLU(),
            #nn.BatchNorm2d(30),
            nn.MaxPool2d((25-3+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(1, 150,(4,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-4+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(1, 150,(5,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-5+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer6 = nn.Sequential(
            nn.Conv2d(1, 150,(6,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-6+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer7 = nn.Sequential(
            nn.Conv2d(1, 150,(7,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-7+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer8 = nn.Sequential(
            nn.Conv2d(1, 150,(8,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-8+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer9 = nn.Sequential(
            nn.Conv2d(1, 150,(9,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-9+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer10 = nn.Sequential(
            nn.Conv2d(1, 150,(10,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-10+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer11 = nn.Sequential(
            nn.Conv2d(1, 150,(11,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-11+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer12 = nn.Sequential(
            nn.Conv2d(1, 150,(12,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-12+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer13 = nn.Sequential(
            nn.Conv2d(1, 150,(13,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-13+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer14 = nn.Sequential(
            nn.Conv2d(1, 150,(14,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-14+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer15 = nn.Sequential(
            nn.Conv2d(1, 150,(15,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-15+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer16 = nn.Sequential(
            nn.Conv2d(1, 150,(16,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-16+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer17 = nn.Sequential(
            nn.Conv2d(1, 150,(17,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-17+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer18 = nn.Sequential(
            nn.Conv2d(1, 150,(18,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-18+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer19 = nn.Sequential(
            nn.Conv2d(1, 150,(19,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-19+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer20 = nn.Sequential(
            nn.Conv2d(1, 150,(20,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-20+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer21 = nn.Sequential(
            nn.Conv2d(1, 150,(21,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-21+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer22 = nn.Sequential(
            nn.Conv2d(1, 150,(22,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-22+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer23 = nn.Sequential(
            nn.Conv2d(1, 150,(23,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-23+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer24 = nn.Sequential(
            nn.Conv2d(1, 150,(24,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-24+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer25 = nn.Sequential(
            nn.Conv2d(1, 150,(25,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-25+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=5, nhead=5), num_layers=6)
        self.linear1 = nn.Linear(in_features=3*25, out_features=30)
        self.linear2 = nn.Linear(in_features=3*25, out_features=30)
        self.decoder = nn.Sequential(
            nn.Linear(in_features=30, out_features=int(150*25/2)),
            nn.ReLU(),
            nn.Linear(in_features=int(150*25/2), out_features=150*25),
            nn.Unflatten(1,(25,150,1)),
            nn.Conv2d(in_channels=25, out_channels=1, kernel_size=(150-25+1,5), padding=(0,4))
        )
    def forward(self, x):
        padding_mask = torch.sum(x.squeeze(dim=1),dim=2) == 0
        x = x.permute(2,1,0,3)
        x = torch.squeeze(x,dim=1)
        x = self.transformer_encoder(src = x,src_key_padding_mask = padding_mask)
        x = torch.unsqueeze(x,dim=1)
        x = x.permute(2,1,0,3)
        f1 = self.layer1(x)
        f2 = self.layer2(x)
        f3 = self.layer3(x)
        f4 = self.layer4(x)
        f5 = self.layer5(x)
        f6 = self.layer6(x)
        f7 = self.layer7(x)
        f8 = self.layer8(x)
        f9 = self.layer9(x)
        f10 = self.layer10(x)
        f11 = self.layer11(x)
        f12 = self.layer12(x)
        f13 = self.layer13(x)
        f14 = self.layer14(x)
        f15 = self.layer15(x)
        f16 = self.layer16(x)
        f17 = self.layer17(x)
        f18 = self.layer18(x)
        f19 = self.layer19(x)
        f20 = self.layer20(x)
        f21 = self.layer21(x)
        f22 = self.layer22(x)
        f23 = self.layer23(x)
        f24 = self.layer24(x)
        f25 = self.layer25(x)
        mu = self.linear1(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,f22,f23,f24,f25),1))
        #logvar = self.linear2(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,f22,f23,f24,f25),1))
        #std = torch.exp(0.5*logvar)
        #eps = torch.randn_like(std)
        #encoded = mu + eps*std
        #encoded = mu
        #decoded = self.decoder(encoded)
        #return encoded, decoded, mu, logvar
        return mu

In [25]:
# 5 cdr3 beta
class cdr3VAEb(nn.Module):
    def __init__(self):
        super(cdr3VAEb, self).__init__()
        self.layer1 = nn.Sequential(  
            nn.Conv2d(1, 150,(1,5)),
            nn.ReLU(),   
            nn.MaxPool2d((25,1)),
            nn.Flatten(), 
            nn.Linear(150, int(150/2)),    
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(1, 150,(2,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-2+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(1, 150,(3,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-3+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(1, 150,(4,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-4+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(1, 150,(5,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-5+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer6 = nn.Sequential(
            nn.Conv2d(1, 150,(6,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-6+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer7 = nn.Sequential(
            nn.Conv2d(1, 150,(7,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-7+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer8 = nn.Sequential(
            nn.Conv2d(1, 150,(8,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-8+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer9 = nn.Sequential(
            nn.Conv2d(1, 150,(9,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-9+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer10 = nn.Sequential(
            nn.Conv2d(1, 150,(10,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-10+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer11 = nn.Sequential(
            nn.Conv2d(1, 150,(11,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-11+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer12 = nn.Sequential(
            nn.Conv2d(1, 150,(12,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-12+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer13 = nn.Sequential(
            nn.Conv2d(1, 150,(13,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-13+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer14 = nn.Sequential(
            nn.Conv2d(1, 150,(14,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-14+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer15 = nn.Sequential(
            nn.Conv2d(1, 150,(15,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-15+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer16 = nn.Sequential(
            nn.Conv2d(1, 150,(16,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-16+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer17 = nn.Sequential(
            nn.Conv2d(1, 150,(17,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-17+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer18 = nn.Sequential(
            nn.Conv2d(1, 150,(18,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-18+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer19 = nn.Sequential(
            nn.Conv2d(1, 150,(19,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-19+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer20 = nn.Sequential(
            nn.Conv2d(1, 150,(20,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-20+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer21 = nn.Sequential(
            nn.Conv2d(1, 150,(21,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-21+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer22 = nn.Sequential(
            nn.Conv2d(1, 150,(22,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-22+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer23 = nn.Sequential(
            nn.Conv2d(1, 150,(23,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-23+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer24 = nn.Sequential(
            nn.Conv2d(1, 150,(24,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-24+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.layer25 = nn.Sequential(
            nn.Conv2d(1, 150,(25,5)),
            nn.ReLU(),
            nn.MaxPool2d((25-25+1,1)),
            nn.Flatten(),
            nn.Linear(150, int(150/2)),
            nn.ReLU(),
            nn.Linear(int(150/2), 3)
        )
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model=5, nhead=5), num_layers=6)
        self.linear1 = nn.Linear(in_features=3*25, out_features=30)
        self.linear2 = nn.Linear(in_features=3*25, out_features=30)
        self.decoder = nn.Sequential(
            nn.Linear(in_features=30, out_features=int(150*25/2)),
            nn.ReLU(),
            nn.Linear(in_features=int(150*25/2), out_features=150*25),
            nn.Unflatten(1,(25,150,1)),
            nn.Conv2d(in_channels=25, out_channels=1, kernel_size=(150-25+1,5), padding=(0,4))
        )
    def forward(self, x):
        padding_mask = torch.sum(x.squeeze(dim=1),dim=2) == 0
        x = x.permute(2,1,0,3)
        x = torch.squeeze(x,dim=1)
        x = self.transformer_encoder(src = x,src_key_padding_mask = padding_mask)
        x = torch.unsqueeze(x,dim=1)
        x = x.permute(2,1,0,3)
        f1 = self.layer1(x)
        f2 = self.layer2(x)
        f3 = self.layer3(x)
        f4 = self.layer4(x)
        f5 = self.layer5(x)
        f6 = self.layer6(x)
        f7 = self.layer7(x)
        f8 = self.layer8(x)
        f9 = self.layer9(x)
        f10 = self.layer10(x)
        f11 = self.layer11(x)
        f12 = self.layer12(x)
        f13 = self.layer13(x)
        f14 = self.layer14(x)
        f15 = self.layer15(x)
        f16 = self.layer16(x)
        f17 = self.layer17(x)
        f18 = self.layer18(x)
        f19 = self.layer19(x)
        f20 = self.layer20(x)
        f21 = self.layer21(x)
        f22 = self.layer22(x)
        f23 = self.layer23(x)
        f24 = self.layer24(x)
        f25 = self.layer25(x)
        mu = self.linear1(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,f22,f23,f24,f25),1))
        #logvar = self.linear2(torch.cat((f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20,f21,f22,f23,f24,f25),1))
        #std = torch.exp(0.5*logvar)
        #eps = torch.randn_like(std)
        #encoded = mu + eps*std
        #encoded = mu
        #decoded = self.decoder(encoded)
        #return encoded, decoded, mu, logvar
        return mu

In [26]:
# def pmhcEncoder(model, source_dataset):
#     x_p = torch.Tensor(peptideMap(source_dataset, aa_dict_atchley, "peptide", 30)).to(DEVICE)
#     x_a = torch.Tensor(mhcMap(source_dataset, "mhca", mhc_dic)).to(DEVICE)
#     x_b = torch.Tensor(mhcMap(source_dataset, "mhcb", mhc_dic)).to(DEVICE)
#     encoded, output = model(x_p, x_a, x_b)
#     return encoded

In [27]:
def tcrEncoder(model, source_dataset, column, padding):
    seq = torch.Tensor(peptideMap(source_dataset, aa_dict_atchley, column, padding)).to(DEVICE)
    #encoded, recon, mu, logvar = model(seq)
    encoded = model(seq)
    encoded[torch.isnan(encoded).all(dim=1)] = 0
    return encoded

In [28]:
# def pmhcEncoding(model, batch_data):
#     batch_data_uniq = batch_data[["peptide","mhca","mhcb"]].drop_duplicates().reset_index(drop=True)
#     embedding_of_batch_uniq = nn.functional.normalize(pmhcEncoder(model, batch_data_uniq))
#     cat_batch_uniq_embedding = pd.concat([batch_data_uniq, pd.DataFrame(embedding_of_batch_uniq.detach().cpu().numpy())], axis=1)
#     embedding_of_batch_data = pd.merge(batch_data[["peptide","mhca","mhcb"]].reset_index(drop=True), cat_batch_uniq_embedding, left_on=["peptide","mhca","mhcb"], right_on=["peptide","mhca","mhcb"], how="left")
#     pmhc_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["peptide","mhca","mhcb"]).values).to(DEVICE)
#     return pmhc_embedding

In [29]:
def pmhcEncoding(batch_data):
    embedding_of_batch_data = pd.merge(batch_data[["peptide","mhca","mhcb"]].reset_index(drop=True), SAVED_PMHC, left_on=["peptide","mhca","mhcb"], right_on=["peptide","mhca","mhcb"], how="left")
    pmhc_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["peptide","mhca","mhcb"]).values,dtype=torch.float32).to(DEVICE)
    return pmhc_embedding

In [30]:
# def vaEncoding(batch_data):
#     #batch_data_uniq = batch_data[["vaseq"]].drop_duplicates().reset_index(drop=True)
#     #embedding_of_batch_uniq = nn.functional.normalize(tcrEncoder(model, batch_data_uniq, "vaseq", 100))
#     #cat_batch_uniq_embedding = pd.concat([batch_data_uniq, pd.DataFrame(embedding_of_batch_uniq.detach().cpu().numpy())], axis=1)
#     embedding_of_batch_data = pd.merge(batch_data[["vaseq"]].reset_index(drop=True), VA_EMBEDDING_SAVED, left_on="vaseq", right_on="vaseq", how="left")
#     va_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["vaseq"]).values,dtype=torch.float32).to(DEVICE)
#     return va_embedding

In [31]:
# def vbEncoding(batch_data):
#     #batch_data_uniq = batch_data[["vbseq"]].drop_duplicates().reset_index(drop=True)
#     #embedding_of_batch_uniq = nn.functional.normalize(tcrEncoder(model, batch_data_uniq, "vbseq", 100))
#     #cat_batch_uniq_embedding = pd.concat([batch_data_uniq, pd.DataFrame(embedding_of_batch_uniq.detach().cpu().numpy())], axis=1)
#     embedding_of_batch_data = pd.merge(batch_data[["vbseq"]].reset_index(drop=True), VB_EMBEDDING_SAVED, left_on="vbseq", right_on="vbseq", how="left")
#     vb_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["vbseq"]).values,dtype=torch.float32).to(DEVICE)
#     return vb_embedding

In [32]:
def vaEncoding(model, batch_data):
    batch_data_uniq = batch_data[["vaseq"]].drop_duplicates().reset_index(drop=True)
    embedding_of_batch_uniq = nn.functional.normalize(tcrEncoder(model, batch_data_uniq, "vaseq", 100))
    cat_batch_uniq_embedding = pd.concat([batch_data_uniq, pd.DataFrame(embedding_of_batch_uniq.detach().cpu().numpy())], axis=1)
    embedding_of_batch_data = pd.merge(batch_data[["vaseq"]].reset_index(drop=True), cat_batch_uniq_embedding, left_on="vaseq", right_on="vaseq", how="left")
    va_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["vaseq"]).values).to(DEVICE)
    return va_embedding

In [33]:
def vbEncoding(model, batch_data):
    batch_data_uniq = batch_data[["vbseq"]].drop_duplicates().reset_index(drop=True)
    embedding_of_batch_uniq = nn.functional.normalize(tcrEncoder(model, batch_data_uniq, "vbseq", 100))
    cat_batch_uniq_embedding = pd.concat([batch_data_uniq, pd.DataFrame(embedding_of_batch_uniq.detach().cpu().numpy())], axis=1)
    embedding_of_batch_data = pd.merge(batch_data[["vbseq"]].reset_index(drop=True), cat_batch_uniq_embedding, left_on="vbseq", right_on="vbseq", how="left")
    vb_embedding = torch.tensor(embedding_of_batch_data.drop(columns=["vbseq"]).values).to(DEVICE)
    return vb_embedding

In [34]:
# input dataset, like vcdr3pmhc
# output Zpmhc * Ztcr
class pMHCTCR70(nn.Module):
    def __init__(self, pmhc=None, va=None, vb=None, cdr3a=None, cdr3b=None, temperature=0.1):
        super(pMHCTCR70, self).__init__()
        
        self.temperature = temperature
        
        self.pmhcmodel = pMHC()
        if pmhc is not None:
            self.pmhcmodel.load_state_dict(torch.load(pmhc,map_location=DEVICE)['net'])
        
        self.vamodel = vGdVAEa()
        if va is not None:
            self.vamodel.load_state_dict(torch.load(va,map_location=DEVICE)['net'])

        self.vbmodel = vGdVAEb()
        if vb is not None:
            self.vbmodel.load_state_dict(torch.load(vb,map_location=DEVICE)['net'])

        self.cdr3amodel = cdr3VAEa()
        if cdr3a is not None:
            self.cdr3amodel.load_state_dict(torch.load(cdr3a,map_location=DEVICE)['net'])

        self.cdr3bmodel = cdr3VAEb()
        if cdr3b is not None:
            self.cdr3bmodel.load_state_dict(torch.load(cdr3b,map_location=DEVICE)['net'])
        
         # Proj for pMHC
        self.Proj1 = nn.Sequential(
            nn.Linear(30, 40),
            nn.ReLU(),
            nn.Linear(40, 50),
            nn.ReLU(),
            nn.Linear(50, 60),
            nn.ReLU(),
            nn.Linear(60, 70)
        )
        
        self.linearTCR = nn.Sequential(
            nn.Linear(70, 70),
            nn.ReLU(),
            nn.Linear(70, 70)
        )
        
        # Proj for TCR dim_in is 5*2+30*2
        self.Proj2 = nn.Sequential(
            nn.Linear(70, 70),
            nn.ReLU(),
            nn.Linear(70, 70)
        )
        
    def forward(self, batch_data):
         
        #pmhc_embedding = pmhcEncoding(self.pmhcmodel, batch_data)
        pmhc_embedding = pmhcEncoding(batch_data)

        va_embedding = vaEncoding(self.vamodel, batch_data)
        #va_embedding = vaEncoding(batch_data)

        vb_embedding = vbEncoding(self.vbmodel, batch_data)
        #vb_embedding = vbEncoding(batch_data)

        cdr3a_embedding = nn.functional.normalize(tcrEncoder(self.cdr3amodel, batch_data, "cdr3a", 25))

        cdr3b_embedding = nn.functional.normalize(tcrEncoder(self.cdr3bmodel, batch_data, "cdr3b", 25))

        tcr_embedding = self.linearTCR(torch.cat((va_embedding, cdr3a_embedding, vb_embedding, cdr3b_embedding),dim=1))

        Zpmhc = F.normalize(self.Proj1(pmhc_embedding))

        Ztcr = F.normalize(self.Proj2(tcr_embedding))

        logits = torch.div(torch.diagonal(torch.mm(Zpmhc,Ztcr.T)),self.temperature)

        return logits

In [35]:
# val with AUROC
# compute AUROC with background TCR for only once
# def val_auroc(dataset, human_alpha, human_beta, mouse_alpha, mouse_beta, model, bg_tcr):
    
#     try:
        
#         dataset_with_background = add_background_tcr(dataset.reset_index(drop=True), human_alpha, human_beta, mouse_alpha, mouse_beta, bg_tcr)
#         results_pct_tmp = Calculate_backgroundtcr(dataset_with_background.sort_values(by=['peptide','mhca','mhcb','vaseq','vbseq','cdr3a','cdr3b']).reset_index(drop=True), model)
#         results_pct_tcr = results_pct_tmp.loc[results_pct_tmp['_data_type']=="validation"]

#     except:
#         print(results_pct)
#         print(len(results_pct_hc1))
#         print(len(results_pct_hc2))
#         print(len(results_pct_mc1))
#         print(len(results_pct_mc2))
        
#         print(len(results_pct_tcr))
#         print(len(results_pct_tcr_hc1))
#         print(len(results_pct_tcr_hc2))
#         print(len(results_pct_tcr_mc1))
#         print(len(results_pct_tcr_mc2))
        
#         print(len(results_pct_pmhc))
#         print(len(results_pct_pmhc_hc1))
#         print(len(results_pct_pmhc_hc2))
#         print(len(results_pct_pmhc_mc1))
#         print(len(results_pct_pmhc_mc2))
    
#     return results_pct_tcr.reset_index(drop=True)

In [36]:
# val with AUROC
# compute AUROC with background TCR for only once
def val_auroc(dataset, human_alpha, human_beta, mouse_alpha, mouse_beta, human_bg_pmhc, mouse_bg_pmhc, model, bg_tcr, bg_pmhc):
    try:
        
        
        
        dataset_with_background = add_background_tcr(dataset.reset_index(drop=True), human_alpha, human_beta, mouse_alpha, mouse_beta, bg_tcr)
        
        
        results_pct_tmp = Calculate_backgroundtcr(dataset_with_background.sort_values(by=['peptide','mhca','mhcb','vaseq','vbseq','cdr3a','cdr3b']).reset_index(drop=True), model)
        
        #results_pct_tcr = results_pct_tmp.loc[results_pct_tmp['_data_type']=="validation"]
        results_pct_tcr = external_rank(results_pct_tmp.loc[results_pct_tmp['_data_type']=="validation"].reset_index(drop=True), results_pct_tmp.loc[results_pct_tmp['_data_type']=="background"].reset_index(drop=True),'_pmhc')
        
        dataset_with_background = add_background_pmhc(dataset.reset_index(drop=True), human_bg_pmhc, mouse_bg_pmhc, bg_pmhc)
        
        results_pct_tmp = Calculate_backgroundpmhc(dataset_with_background.sort_values(by=['peptide','mhca','mhcb','vaseq','vbseq','cdr3a','cdr3b']).reset_index(drop=True), model)
        
        #results_pct_pmhc = results_pct_tmp.loc[results_pct_tmp['_data_type']=="validation"]
        results_pct_pmhc = external_rank(results_pct_tmp.loc[results_pct_tmp['_data_type']=="validation"].reset_index(drop=True), results_pct_tmp.loc[results_pct_tmp['_data_type']=="background"].reset_index(drop=True),'_tcr')
        
        tmp_test=pd.concat([results_pct_tcr[['_pmhctcr']].reset_index(drop=True).rename(columns={'_pmhctcr':'_pmhctcr1'}),results_pct_pmhc[["_pmhctcr"]].reset_index(drop=True).rename(columns={'_pmhctcr':'_pmhctcr2'})],axis=1)
        #return(tmp_test)
        if tmp_test['_pmhctcr1'].equals(tmp_test['_pmhctcr2']):

            #results_pct_tcr = results_pct_tcr.rename(columns={"_value": "_value_bgTCR","percentile_rank": "percentile_rank_bgTCR","percentile_rank_asc": "percentile_rank_asc_bgTCR"})
            results_pct_tcr = results_pct_tcr.rename(columns={"percentile_rank": "percentile_rank_bgTCR","percentile_rank_asc": "percentile_rank_asc_bgTCR","external_rank_1":"external_rank_1_bgTCR","external_rank_2":"external_rank_2_bgTCR"})

            #results_pct_pmhc = results_pct_pmhc.rename(columns={"_value": "_value_bgPMHC","percentile_rank": "percentile_rank_bgPMHC","percentile_rank_asc": "percentile_rank_asc_bgPMHC"})
            results_pct_pmhc = results_pct_pmhc.rename(columns={"percentile_rank": "percentile_rank_bgPMHC","percentile_rank_asc": "percentile_rank_asc_bgPMHC", 'external_rank_1':'external_rank_1_bgPMHC', 'external_rank_2':'external_rank_2_bgPMHC'})
            
            results_pct= pd.concat([results_pct_tcr.reset_index(drop=True),results_pct_pmhc[["percentile_rank_bgPMHC","percentile_rank_asc_bgPMHC","external_rank_1_bgPMHC","external_rank_2_bgPMHC"]].reset_index(drop=True)],axis=1)

        else:
            exit()
        
        #results_pct['_value_bgAVG'] = (results_pct['_value_bgTCR'].reset_index(drop=True) + results_pct['_value_bgPMHC'].reset_index(drop=True))/2
        results_pct['percentile_rank_bgAVG'] = (results_pct['percentile_rank_bgTCR'].reset_index(drop=True) + results_pct['percentile_rank_bgPMHC'].reset_index(drop=True))/2
        results_pct['percentile_rank_asc_bgAVG'] = (results_pct['percentile_rank_asc_bgTCR'].reset_index(drop=True) + results_pct['percentile_rank_asc_bgPMHC'].reset_index(drop=True))/2
        
        results_pct['external_rank_bgAVG'] = (results_pct['external_rank_2_bgTCR'].reset_index(drop=True) + results_pct['external_rank_2_bgPMHC'].reset_index(drop=True))/2
        results_pct['external_rank_asc_bgAVG'] = (results_pct['external_rank_1_bgTCR'].reset_index(drop=True) + results_pct['external_rank_1_bgPMHC'].reset_index(drop=True))/2
        
        #results_pct['_value_bgMAX'] = results_pct[['_value_bgTCR','_value_bgPMHC']].max(axis=1)
        results_pct['percentile_rank_bgMAX'] = results_pct[['percentile_rank_bgTCR','percentile_rank_bgPMHC']].max(axis=1)
        results_pct['percentile_rank_asc_bgMAX'] = results_pct[['percentile_rank_asc_bgTCR','percentile_rank_asc_bgPMHC']].max(axis=1)
        
        results_pct['external_rank_bgMAX'] = results_pct[['external_rank_2_bgTCR','external_rank_2_bgPMHC']].max(axis=1)
        results_pct['external_rank_asc_bgMAX'] = results_pct[['external_rank_1_bgTCR','external_rank_1_bgPMHC']].max(axis=1)
        
        results_pct['percentile_rank_bgMIN'] = results_pct[['percentile_rank_bgTCR','percentile_rank_bgPMHC']].min(axis=1)
        results_pct['percentile_rank_asc_bgMIN'] = results_pct[['percentile_rank_asc_bgTCR','percentile_rank_asc_bgPMHC']].min(axis=1)
        
        results_pct['external_rank_bgMIN'] = results_pct[['external_rank_2_bgTCR','external_rank_2_bgPMHC']].min(axis=1)
        results_pct['external_rank_asc_bgMIN'] = results_pct[['external_rank_1_bgTCR','external_rank_1_bgPMHC']].min(axis=1)
        
        
        
#         auroc_avg_all = roc_auc_score(results_pct["label"], results_pct["percentile_rank_asc_bgAVG"])

#         auroc_max_all = roc_auc_score(results_pct["label"], results_pct["percentile_rank_asc_bgMAX"])

#         auroc_tcr_all = roc_auc_score(results_pct_tcr["label"], results_pct_tcr["percentile_rank_asc_bgTCR"])     

#         auroc_pmhc_all = roc_auc_score(results_pct_pmhc["label"], results_pct_pmhc["percentile_rank_asc_bgPMHC"])
        
#         #logits
#         auc_logits = roc_auc_score(results_pct_tcr["label"], results_pct["_value"])
        
              
        
    except:
        print(results_pct)
        print(len(results_pct_hc1))
        print(len(results_pct_hc2))
        print(len(results_pct_mc1))
        print(len(results_pct_mc2))
        
        print(len(results_pct_tcr))
        print(len(results_pct_tcr_hc1))
        print(len(results_pct_tcr_hc2))
        print(len(results_pct_tcr_mc1))
        print(len(results_pct_tcr_mc2))
        
        print(len(results_pct_pmhc))
        print(len(results_pct_pmhc_hc1))
        print(len(results_pct_pmhc_hc2))
        print(len(results_pct_pmhc_mc1))
        print(len(results_pct_pmhc_mc2))
    
    return results_pct
    #return results_pct, auroc_avg_all, auroc_max_all, auroc_tcr_all, auroc_pmhc_all, auc_logits
    #return results_pct, auroc_all, auroc_hc1, auroc_hc2, auroc_m, auroc_mc1, auroc_mc2, auroc_tcr_all, auroc_tcr_hc1, auroc_tcr_hc2, auroc_tcr_m, auroc_tcr_mc1, auroc_tcr_mc2, auroc_pmhc_all, auroc_pmhc_hc1, auroc_pmhc_hc2, auroc_pmhc_m, auroc_pmhc_mc1, auroc_pmhc_mc2
    #return auroc_all, auroc_hc1, auroc_hc2, auroc_m, auroc_mc1, auroc_mc2, median_pos_rank, mean_pos_rank, median_neg_rank, mean_neg_rank

In [37]:
# def rank_in_column_b_ascending(value, column_b):
#     return (column_b < value).sum()/len(column_b)

# def rank_in_column_b_descending(value, column_b):
#     return (column_b > value).sum()/len(column_b)

# def external_rank(data_val, data_back):
   
#     data_val['external_rank_1'] = data_val['_value'].apply(rank_in_column_b_ascending, column_b=data_back['_value'])
#     data_val['external_rank_2'] = data_val['_value'].apply(rank_in_column_b_descending, column_b=data_back['_value'])
    
#     return(data_val)

In [38]:
# def rank_in_column_b_ascending(row_a, data_b, key):
#     subset=data_b[data_b[key]==row_a[key]]
#     #return (data_b._value[data_b[key]==row_a[key]] < row_a['_value']).sum()/len(data_b[data_b[key[0]]==row_a[key[0]]])
#     return (subset._value < row_a['_value']).sum()/len(subset)

# def rank_in_column_b_descending(row_a, data_b, key):
#     subset=data_b[data_b[key]==row_a[key]]
#     #return (data_b._value[data_b[key]==row_a[key]] > row_a['_value']).sum()/len(data_b[data_b[key[0]]==row_a[key[0]]])
#     return (subset._value > row_a['_value']).sum()/len(subset)

# def external_rank(data_val, data_back, key_column):
#     #print(key_column)
    
#     data_val['external_rank_1'] = data_val[[key_column,'_value']].parallel_apply(rank_in_column_b_ascending, data_b=data_back[[key_column,'_value']], key=key_column, axis=1)
#     data_val['external_rank_2'] = data_val[[key_column,'_value']].parallel_apply(rank_in_column_b_descending, data_b=data_back[[key_column,'_value']], key=key_column, axis=1)
    
#     return(data_val)

In [39]:
def rank_in_column_b(row_a, data_b, key):
    subset=data_b[data_b[key]==row_a[key]]
    return (subset._value < row_a['_value']).sum()/len(subset), (subset._value > row_a['_value']).sum()/len(subset)

def external_rank(data_val, data_back, key_column):
    data_val[['external_rank_1','external_rank_2']] = data_val[[key_column,'_value']].parallel_apply(rank_in_column_b, data_b=data_back[[key_column,'_value']], key=key_column, axis=1, result_type='expand')
    return(data_val)

In [40]:
@background(max_prefetch=80000)
def iterate_minibatches(dataset, num_dp):
    i = 0
    while i < len(dataset):
        batch_data = dataset[i:(i+num_dp)]
        i = i+num_dp
        yield batch_data.reset_index(drop=True)

In [41]:
def Calculate_backgroundtcr(dataset, model):
    model.eval()
    results = pd.DataFrame()
    #num_dataset = dataset.shape[0]
    
    with torch.no_grad():
        #for loop, batch in enumerate(range(0, num_dataset, VAL_BATCH_SIZE_bgTCR)):
        for batch_data in iterate_minibatches(dataset,VAL_BATCH_SIZE_bgTCR):
            #print(len(batch_data))            
            #batch_data = dataset[batch:(batch+VAL_BATCH_SIZE_bgTCR)].reset_index(drop=True)
                         
            model_value = model(batch_data[["peptide","mhca","mhcb",'vaseq','vbseq','cdr3a','cdr3b']])
            
            #results = pd.concat([results, pd.concat([batch_data, pd.DataFrame(classified.detach().cpu().numpy()[:,1],columns=['_value'])],axis=1)])
            results = pd.concat([results, pd.concat([batch_data, pd.DataFrame(model_value.detach().cpu().numpy(),columns=['_value'])],axis=1)])
            
    results = results.reset_index(drop=True)
    
    pct = results[['_pmhc','_value']].groupby(['_pmhc']).rank(ascending=False,pct=True).rename(columns={"_value": "percentile_rank"}) # resultes for user, smaller means stronger binding

    pct_asc = results[['_pmhc','_value']].groupby(['_pmhc']).rank(ascending=True,pct=True).rename(columns={"_value": "percentile_rank_asc"}) # used to calculate auroc, bigger means stronger binding
    
    results_pct = pd.concat([results, pct, pct_asc],axis=1)
    
    return results_pct

In [42]:
def Calculate_backgroundpmhc(dataset, model):
    model.eval()
    results = pd.DataFrame()
    #num_dataset = dataset.shape[0]
    
    with torch.no_grad():
        #for loop, batch in enumerate(range(0, num_dataset, VAL_BATCH_SIZE_bgPMHC)):
        for batch_data in iterate_minibatches(dataset,VAL_BATCH_SIZE_bgPMHC):
            #print(len(batch_data))
            #batch_data = dataset[batch:(batch+VAL_BATCH_SIZE_bgPMHC)].reset_index(drop=True)
                                       
            model_value = model(batch_data[["peptide","mhca","mhcb",'vaseq','vbseq','cdr3a','cdr3b']])
            
            results = pd.concat([results, pd.concat([batch_data, pd.DataFrame(model_value.detach().cpu().numpy(),columns=['_value'])],axis=1)])
            #results = pd.concat([results, pd.concat([batch_data, pd.DataFrame(classified.detach().cpu().numpy()[:,1],columns=['_value'])],axis=1)])
            
            
    results = results.reset_index(drop=True)
    
    pct = results[['_tcr','_value']].groupby(['_tcr']).rank(ascending=False,pct=True).rename(columns={"_value": "percentile_rank"}) # resultes for user, smaller means stronger binding

    pct_asc = results[['_tcr','_value']].groupby(['_tcr']).rank(ascending=True,pct=True).rename(columns={"_value": "percentile_rank_asc"}) # used to calculate auroc, bigger means stronger binding
    
    results_pct = pd.concat([results, pct, pct_asc],axis=1)
    
    return results_pct

In [43]:
saved_pmhc=pd.read_csv('/project/DPDS/Wang_lab/shared/pMTnet_v2/v2_5/data/pmhc_embedding.csv', header=0, sep=",")
SAVED_PMHC=saved_pmhc.drop(columns=["pMHC_SPECIES","pmhc_id"])

In [44]:
#VA_EMBEDDING_SAVED=pd.read_csv('/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/vgene_alpha_embedding.csv', header=0, sep=",")

In [45]:
#VB_EMBEDDING_SAVED=pd.read_csv('/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/vgene_beta_embedding.csv', header=0, sep=",")

In [46]:
# read background TCR
tcr_human_alpha = preprocessTCR("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/backgroundTCR/human_alpha.txt")
tcr_human_beta = preprocessTCR("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/backgroundTCR/human_beta.txt")
tcr_mouse_alpha = preprocessTCR("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/backgroundTCR/mouse_alpha.txt")
tcr_mouse_beta = preprocessTCR("/project/DPDS/Wang_lab/shared/pMTnet_v2/data/backgroundTCR/mouse_beta.txt")

Number of rows in raw dataset: 822103
Number of rows in this dataset after dropping NA: 822103
Number of rows in processed dataset: 822103
Number of rows in raw dataset: 5233069
Number of rows in this dataset after dropping NA: 5233069
Number of rows in processed dataset: 5233069
Number of rows in raw dataset: 913842
Number of rows in this dataset after dropping NA: 913842
Number of rows in processed dataset: 913842
Number of rows in raw dataset: 7589029
Number of rows in this dataset after dropping NA: 7589029
Number of rows in processed dataset: 7589029


In [47]:
tcr_human_alpha = tcr_human_alpha[['vaseq','cdr3a']]
tcr_human_beta = tcr_human_beta[['vbseq','cdr3b']]
tcr_mouse_alpha = tcr_mouse_alpha[['vaseq','cdr3a']]
tcr_mouse_beta = tcr_mouse_beta[['vbseq','cdr3b']]

In [48]:
bg_pmhc =preprocesspMHC2("/project/DPDS/Wang_lab/shared/pMTnet_v2/tcr_promiscuity/data/iedb_background_pmhc.csv")
human_bg_pmhc=bg_pmhc.loc[(bg_pmhc['class']=="hc1") | (bg_pmhc['class']=="hc2"),("peptide","alpha_allele","beta_allele")]
mouse_bg_pmhc=bg_pmhc.loc[(bg_pmhc['class']=="mc1") | (bg_pmhc['class']=="mc2"),("peptide","alpha_allele","beta_allele")]
human_bg_pmhc = human_bg_pmhc.rename(columns={'alpha_allele':'mhca','beta_allele':'mhcb'})
mouse_bg_pmhc = mouse_bg_pmhc.rename(columns={'alpha_allele':'mhca','beta_allele':'mhcb'})

Number of rows in raw dataset: 925136
Number of rows in this dataset after dropping NA: 925136
Number of rows in processed dataset: 925136


In [49]:
DEVICE = torch.device("cuda:"+str(CUDA) if torch.cuda.is_available() else "cpu")
print("used device is:" + str(DEVICE))

used device is:cuda:0


In [50]:
def read_dataset(path,septext):
    dataset=preprocess(path,mhc_dic, "mhca", "mhcb",septext)
    dataset = dataset.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
    dataset = dataset[['peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','label']]
    dataset['_pmhc'] = dataset['peptide']+"_"+dataset['mhca']+"_"+dataset['mhcb']+"_"+dataset['pMHC_SPECIES']
    dataset['_tcr'] = dataset['cdr3a']+"_"+dataset['cdr3b']+"_"+dataset['vaseq']+"_"+dataset['vbseq']+"_"+dataset['TCR_SPECIES']
    dataset['_pmhctcr'] = dataset['_pmhc']+"_"+dataset['_tcr']
    dataset = dataset.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
    return(dataset)

In [51]:
validation_dataset1=preprocess_validation(VAL_PATH1,"\t")
validation_dataset1 = validation_dataset1.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset1 = validation_dataset1[['data_file','peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','valrandn','label']]
validation_dataset1['_pmhc'] = validation_dataset1['peptide']+"_"+validation_dataset1['mhca']+"_"+validation_dataset1['mhcb']+"_"+validation_dataset1['pMHC_SPECIES']
validation_dataset1['_tcr'] = validation_dataset1['cdr3a']+"_"+validation_dataset1['cdr3b']+"_"+validation_dataset1['vaseq']+"_"+validation_dataset1['vbseq']+"_"+validation_dataset1['TCR_SPECIES']
validation_dataset1['_pmhctcr'] = validation_dataset1['_pmhc']+"_"+validation_dataset1['_tcr']
validation_dataset1 = validation_dataset1.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
print(len(validation_dataset1))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_tcr_val5neg_train50neg_maxEpoch70/validation.txt
Number of rows in raw dataset: 4716
Number of rows in processed dataset: 4716
4716


In [52]:
validation_dataset2_1=preprocess_validation(VAL_PATH2_1,"\t")
validation_dataset2_1 = validation_dataset2_1.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset2_1 = validation_dataset2_1[['data_file','peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','valrandn','label']]
validation_dataset2_1['_pmhc'] = validation_dataset2_1['peptide']+"_"+validation_dataset2_1['mhca']+"_"+validation_dataset2_1['mhcb']+"_"+validation_dataset2_1['pMHC_SPECIES']
validation_dataset2_1['_tcr'] = validation_dataset2_1['cdr3a']+"_"+validation_dataset2_1['cdr3b']+"_"+validation_dataset2_1['vaseq']+"_"+validation_dataset2_1['vbseq']+"_"+validation_dataset2_1['TCR_SPECIES']
validation_dataset2_1['_pmhctcr'] = validation_dataset2_1['_pmhc']+"_"+validation_dataset2_1['_tcr']
validation_dataset2_1 = validation_dataset2_1.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
print(len(validation_dataset2_1))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_pmhc_val5neg_train50neg_maxEpoch70/validation.txt
Number of rows in raw dataset: 4716
Number of rows in processed dataset: 4716
4716


In [53]:
validation_dataset2_2=preprocess_validation(VAL_PATH2_2,"\t")
validation_dataset2_2 = validation_dataset2_2.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset2_2 = validation_dataset2_2[['data_file','peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','valrandn','label']]
validation_dataset2_2['_pmhc'] = validation_dataset2_2['peptide']+"_"+validation_dataset2_2['mhca']+"_"+validation_dataset2_2['mhcb']+"_"+validation_dataset2_2['pMHC_SPECIES']
validation_dataset2_2['_tcr'] = validation_dataset2_2['cdr3a']+"_"+validation_dataset2_2['cdr3b']+"_"+validation_dataset2_2['vaseq']+"_"+validation_dataset2_2['vbseq']+"_"+validation_dataset2_2['TCR_SPECIES']
validation_dataset2_2['_pmhctcr'] = validation_dataset2_2['_pmhc']+"_"+validation_dataset2_2['_tcr']
validation_dataset2_2 = validation_dataset2_2.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
print(len(validation_dataset2_2))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_p_val5neg_train50neg_maxEpoch70/validation.txt
Number of rows in raw dataset: 4691
Number of rows in processed dataset: 4691
4691


In [54]:
validation_dataset3=preprocess_validation(VAL_PATH3,"\t")
validation_dataset3 = validation_dataset3.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset3 = validation_dataset3[['peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','label']]
validation_dataset3['_pmhc'] = validation_dataset3['peptide']+"_"+validation_dataset3['mhca']+"_"+validation_dataset3['mhcb']+"_"+validation_dataset3['pMHC_SPECIES']
validation_dataset3['_tcr'] = validation_dataset3['cdr3a']+"_"+validation_dataset3['cdr3b']+"_"+validation_dataset3['vaseq']+"_"+validation_dataset3['vbseq']+"_"+validation_dataset3['TCR_SPECIES']
validation_dataset3['_pmhctcr'] = validation_dataset3['_pmhc']+"_"+validation_dataset3['_tcr']
validation_dataset3 = validation_dataset3.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
print(len(validation_dataset3))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/mar11_change_pmhc_tcr_val5neg_train50neg_maxEpoch70/validation.txt
Number of rows in raw dataset: 4716
Number of rows in processed dataset: 4716
4716


In [55]:
validation_dataset4=preprocess_validation(VAL_PATH4,"\t")
validation_dataset4 = validation_dataset4.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset4 = validation_dataset4[['peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','label']]
validation_dataset4['_pmhc'] = validation_dataset4['peptide']+"_"+validation_dataset4['mhca']+"_"+validation_dataset4['mhcb']+"_"+validation_dataset4['pMHC_SPECIES']
validation_dataset4['_tcr'] = validation_dataset4['cdr3a']+"_"+validation_dataset4['cdr3b']+"_"+validation_dataset4['vaseq']+"_"+validation_dataset4['vbseq']+"_"+validation_dataset4['TCR_SPECIES']
validation_dataset4['_pmhctcr'] = validation_dataset4['_pmhc']+"_"+validation_dataset4['_tcr']
validation_dataset4 = validation_dataset4.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)
print(len(validation_dataset4))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/vcdr3pmhc/feb23_1tcr_more_pmhc_pseudoVgene_upHc2Mouse_val5neg_train50neg_maxEpoch70/validation_hormad1.txt
Number of rows in raw dataset: 51
Number of rows in processed dataset: 51
51


In [56]:
validation_dataset5=preprocess_validation(VAL_PATH5,",")
validation_dataset5 = validation_dataset5.sort_values(by=['peptide', 'mhca', 'mhcb','pMHC_SPECIES'])
validation_dataset5 = validation_dataset5[['peptide', 'mhca', 'mhcb','pMHC_SPECIES','vaseq','cdr3a','vbseq','cdr3b','TCR_SPECIES','label']]
validation_dataset5['_pmhc'] = validation_dataset5['peptide']+"_"+validation_dataset5['mhca']+"_"+validation_dataset5['mhcb']+"_"+validation_dataset5['pMHC_SPECIES']
validation_dataset5['_tcr'] = validation_dataset5['cdr3a']+"_"+validation_dataset5['cdr3b']+"_"+validation_dataset5['vaseq']+"_"+validation_dataset5['vbseq']+"_"+validation_dataset5['TCR_SPECIES']
validation_dataset5['_pmhctcr'] = validation_dataset5['_pmhc']+"_"+validation_dataset5['_tcr']
validation_dataset5 = validation_dataset5.drop_duplicates(subset="_pmhctcr").reset_index(drop=True)

validation_dataset5['tcr_no']=np.NAN
validation_dataset5.cdr3b.unique()
validation_dataset5.loc[validation_dataset5.cdr3b=='CASRRFGDTEAFF','tcr_no']='tcr1'
validation_dataset5.loc[validation_dataset5.cdr3b=='CASSIKASSYNEQFF','tcr_no']='tcr2'
validation_dataset5.loc[validation_dataset5.cdr3b=='CASSAWGGNQPQHF','tcr_no']='tcr3'
validation_dataset5.loc[validation_dataset5.cdr3b=='CSARDLGGDTQYF','tcr_no']='tcr4'
validation_dataset5.loc[validation_dataset5.cdr3b=='CSARSWGSETQYF','tcr_no']='tcr8.1'
validation_dataset5.loc[validation_dataset5.cdr3b=='CASRPLGEETQYF','tcr_no']='tcr8.2'

print(len(validation_dataset5))

Processing: /project/DPDS/Wang_lab/shared/pMTnet_v2/data/koelle/Koelle_input.csv
Number of rows in raw dataset: 42
Number of rows in processed dataset: 42
42


/tmp/ipykernel_252962/3102061388.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'tcr1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  validation_dataset5.loc[validation_dataset5.cdr3b=='CASRRFGDTEAFF','tcr_no']='tcr1'


In [57]:
# #original model
# k=0
# ensembl_results1=pd.DataFrame()
# ensembl_results21=pd.DataFrame()
# ensembl_results22=pd.DataFrame()
# ensembl_results3=pd.DataFrame()
# ensembl_results4=pd.DataFrame()
# ensembl_results5=pd.DataFrame()

# for j in ORIGINAL_MODEL:
#     k=k+1

#     checkpoint = torch.load(j,map_location=DEVICE)
    
#     CLmodel = pMHCTCR70().to(DEVICE)
#     CLmodel.load_state_dict(checkpoint['cl'])
#     #CLmodel = torch.compile(CLmodel)

#     for name, p in CLmodel.named_parameters():
#         if "pmhc" in name or "va" in name or "vb" in name or "cdr3a" in name or "cdr3b" in name:
#             p.requires_grad = False

#     results_pct1 =val_auroc(validation_dataset1, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
#     results_pct21 =val_auroc(validation_dataset2_1, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
#     results_pct22 =val_auroc(validation_dataset2_2, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
#     results_pct3 =val_auroc(validation_dataset3, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
#     results_pct4 =val_auroc(validation_dataset4, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY4,N_BACKGROUND_PMHC_ONLY4)
#     results_pct5 =val_auroc(validation_dataset5, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY5,N_BACKGROUND_PMHC_ONLY5)

#     if k==1:
#         ensembl_results1=results_pct1.iloc[:,16:37].reset_index(drop=True)
        
#         ensembl_results21=results_pct21.iloc[:,16:37].reset_index(drop=True)
        
#         ensembl_results22=results_pct22.iloc[:,16:37].reset_index(drop=True)
        
#         ensembl_results3=results_pct3.iloc[:,14:35].reset_index(drop=True)
        
#         ensembl_results4=results_pct4.iloc[:,14:35].reset_index(drop=True)
        
#         ensembl_results5=results_pct5.iloc[:,15:36].reset_index(drop=True)
        
#     else:
#         ensembl_results1=ensembl_results1+results_pct1.iloc[:,16:37].reset_index(drop=True)
#         ensembl_results21=ensembl_results21+results_pct21.iloc[:,16:37].reset_index(drop=True)
#         ensembl_results22=ensembl_results22+results_pct22.iloc[:,16:37].reset_index(drop=True)
#         ensembl_results3=ensembl_results3+results_pct3.iloc[:,14:35].reset_index(drop=True)
#         ensembl_results4=ensembl_results4+results_pct4.iloc[:,14:35].reset_index(drop=True)
#         ensembl_results5=ensembl_results5+results_pct5.iloc[:,15:36].reset_index(drop=True)

In [58]:
# print(roc_auc_score(results_pct1["label"], ensembl_results1['_value']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgMIN']/k))

In [59]:
# print(roc_auc_score(results_pct21["label"], ensembl_results21['_value']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgMIN']/k))

In [60]:
# print(roc_auc_score(results_pct22["label"], ensembl_results22['_value']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgMIN']/k))

In [61]:
# print(roc_auc_score(results_pct3["label"], ensembl_results3['_value']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgMIN']/k))

In [62]:
# print(roc_auc_score(results_pct4["label"], ensembl_results4['_value']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgMIN']/k))

In [63]:
# print(roc_auc_score(results_pct5["label"], ensembl_results5['_value']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgMIN']/k))

In [ ]:
#new model
k=0
#ensembl_results1=pd.DataFrame()
#ensembl_results21=pd.DataFrame()
#ensembl_results22=pd.DataFrame()
#ensembl_results3=pd.DataFrame()
ensembl_results4=pd.DataFrame()
ensembl_results5=pd.DataFrame()

for j in MODEL_LOADED:
    k=k+1

    checkpoint = torch.load(j,map_location=DEVICE)
    
    CLmodel = pMHCTCR70().to(DEVICE)
    CLmodel.load_state_dict(checkpoint['cl'])
    

    for name, p in CLmodel.named_parameters():
        if "pmhc" in name or "va" in name or "vb" in name or "cdr3a" in name or "cdr3b" in name:
            p.requires_grad = False

    #results_pct1 =val_auroc(validation_dataset1, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
    #results_pct21 =val_auroc(validation_dataset2_1, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
    #results_pct22 =val_auroc(validation_dataset2_2, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
    #results_pct3 =val_auroc(validation_dataset3, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY,N_BACKGROUND_PMHC_ONLY)
    results_pct4 =val_auroc(validation_dataset4, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY4,N_BACKGROUND_PMHC_ONLY4)
    results_pct5 =val_auroc(validation_dataset5, tcr_human_alpha, tcr_human_beta, tcr_mouse_alpha, tcr_mouse_beta, human_bg_pmhc, mouse_bg_pmhc,CLmodel,N_BACKGROUND_TCR_ONLY5,N_BACKGROUND_PMHC_ONLY5)

    if k==1:
        #ensembl_results1=results_pct1.iloc[:,16:37].reset_index(drop=True)
        
        #ensembl_results21=results_pct21.iloc[:,16:37].reset_index(drop=True)
        
        #ensembl_results22=results_pct22.iloc[:,16:37].reset_index(drop=True)
        
        #ensembl_results3=results_pct3.iloc[:,14:35].reset_index(drop=True)
        
        ensembl_results4=results_pct4.iloc[:,14:35].reset_index(drop=True)
        
        ensembl_results5=results_pct5.iloc[:,15:36].reset_index(drop=True)
        
    else:
        #ensembl_results1=ensembl_results1+results_pct1.iloc[:,16:37].reset_index(drop=True)
        #ensembl_results21=ensembl_results21+results_pct21.iloc[:,16:37].reset_index(drop=True)
        #ensembl_results22=ensembl_results22+results_pct22.iloc[:,16:37].reset_index(drop=True)
        #ensembl_results3=ensembl_results3+results_pct3.iloc[:,14:35].reset_index(drop=True)
        ensembl_results4=ensembl_results4+results_pct4.iloc[:,14:35].reset_index(drop=True)
        ensembl_results5=ensembl_results5+results_pct5.iloc[:,15:36].reset_index(drop=True)

In [72]:
k

6

In [ ]:
# print(roc_auc_score(results_pct1["label"], ensembl_results1['_value']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct1["label"], ensembl_results1['external_rank_asc_bgMIN']/k))

In [ ]:
# print(roc_auc_score(results_pct21["label"], ensembl_results21['_value']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct21["label"], ensembl_results21['external_rank_asc_bgMIN']/k))

In [ ]:
# print(roc_auc_score(results_pct22["label"], ensembl_results22['_value']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct22["label"], ensembl_results22['external_rank_asc_bgMIN']/k))

In [ ]:
# print(roc_auc_score(results_pct3["label"], ensembl_results3['_value']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgTCR']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgPMHC']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['percentile_rank_asc_bgMIN']/k))
# print('external')
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_1_bgTCR']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_1_bgPMHC']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgAVG']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgMAX']/k))
# print(roc_auc_score(results_pct3["label"], ensembl_results3['external_rank_asc_bgMIN']/k))

In [ ]:
print(roc_auc_score(results_pct4["label"], ensembl_results4['_value']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgTCR']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgPMHC']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgAVG']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgMAX']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['percentile_rank_asc_bgMIN']/k))
print('external')
print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_1_bgTCR']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_1_bgPMHC']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgAVG']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgMAX']/k))
print(roc_auc_score(results_pct4["label"], ensembl_results4['external_rank_asc_bgMIN']/k))

In [ ]:
print(roc_auc_score(results_pct5["label"], ensembl_results5['_value']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgTCR']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgPMHC']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgAVG']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgMAX']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['percentile_rank_asc_bgMIN']/k))
print('external')
print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_1_bgTCR']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_1_bgPMHC']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgAVG']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgMAX']/k))
print(roc_auc_score(results_pct5["label"], ensembl_results5['external_rank_asc_bgMIN']/k))

In [83]:
final4=pd.concat([results_pct4.iloc[:,0:14],ensembl_results4/k],axis=1)

In [ ]:
print(roc_auc_score(final4["label"], final4['external_rank_1_bgTCR']))
print(roc_auc_score(final4["label"], final4['external_rank_1_bgPMHC']))
print(roc_auc_score(final4["label"], final4['external_rank_asc_bgAVG']))
print(roc_auc_score(final4["label"], final4['external_rank_asc_bgMAX']))
print(roc_auc_score(final4["label"], final4['external_rank_asc_bgMIN']))

In [76]:
final=pd.concat([results_pct5.iloc[:,0:14],ensembl_results5/k],axis=1)

In [ ]:
print(roc_auc_score(final["label"], final['external_rank_1_bgTCR']))
print(roc_auc_score(final["label"], final['external_rank_1_bgPMHC']))
print(roc_auc_score(final["label"], final['external_rank_asc_bgAVG']))
print(roc_auc_score(final["label"], final['external_rank_asc_bgMAX']))
print(roc_auc_score(final["label"], final['external_rank_asc_bgMIN']))